In [7]:
#モデル定義
class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),  # (1 → 32)
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),  # (32 → 64)
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),  # (64 → 128)
            nn.AdaptiveAvgPool2d((1, 1))  # → (B*T, 128, 1, 1)
        )
        self.fc1d = nn.Sequential(
            nn.Conv1d(129, 128, 3, padding=1), nn.ReLU(),  # 128 from CNN + 1 speed
            nn.Conv1d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, imgs, speeds):
        B, T, C, H, W = imgs.shape
        imgs = imgs.view(B * T, C, H, W)
        feats = self.cnn(imgs).view(B, T, -1)  # → (B, T, 128)
        speeds = speeds.unsqueeze(-1)  # → (B, T, 1)
        concat = torch.cat([feats, speeds], dim=2).permute(0, 2, 1)  # → (B, 129, T)
        out = self.fc1d(concat)
        return out.squeeze(1)


In [1]:
import os
import shutil
import numpy as np
from tqdm import tqdm

# 入力元フォルダ
source_dir = "right_images"

# 出力先フォルダ
train_dir = "validation/train_split"
val_dir = "validation/val_split"

# シーンID一覧（000〜738まで存在するフォルダのみ対象）
scene_ids = sorted([
    sid for sid in os.listdir(source_dir)
    if sid.isdigit() and len(sid) == 3 and os.path.isdir(os.path.join(source_dir, sid))
])

# ランダムシャッフル → 8:2 分割
np.random.seed(42)
np.random.shuffle(scene_ids)
split_idx = int(len(scene_ids) * 0.8)
train_ids = scene_ids[:split_idx]
val_ids = scene_ids[split_idx:]

# 出力先フォルダ作成
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# シーンをコピーする関数
def copy_scenes(scene_list, output_root):
    for sid in tqdm(scene_list, desc=f"Copying to {output_root}"):
        src = os.path.join(source_dir, sid)
        dst = os.path.join(output_root, sid)
        shutil.copytree(src, dst, dirs_exist_ok=True)

# 実行
copy_scenes(train_ids, train_dir)
copy_scenes(val_ids, val_dir)

print(f"✅ 完了：{len(train_ids)} scenes to train, {len(val_ids)} scenes to val")


Copying to validation/val_split: 100%|██████████| 148/148 [00:55<00:00,  2.64it/s]

✅ 完了：589 scenes to train, 148 scenes to val


In [4]:


import os
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import Adam
from tqdm import tqdm


class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc1d = nn.Sequential(
            nn.Conv1d(33, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, 1)
        )

    def forward(self, imgs, speeds):  
        B, T, C, H, W = imgs.shape
        imgs = imgs.view(B * T, C, H, W)
        feats = self.cnn(imgs).view(B, T, -1)  
        speeds = speeds.unsqueeze(-1)  
        concat = torch.cat([feats, speeds], dim=2).permute(0, 2, 1)  
        out = self.fc1d(concat)  
        return out.squeeze(1)


class SpeedDataset(Dataset):
    def __init__(self, crop_root, annot_root):
        self.items = []
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ])
        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue
            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) < 15: continue

            imgs = []
            for i in range(15):
                img_path = os.path.join(crop_dir, files[i])
                img = Image.open(img_path).convert("L")
                imgs.append(self.transform(img))
            if len(imgs) < 15: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)
                if len(ann['sequence']) < 15: continue
                own_speed = torch.tensor([f['OwnSpeed'] for f in ann['sequence'][:15]], dtype=torch.float32)
                tgt_speed = torch.tensor([f['TgtSpeed_ref'] for f in ann['sequence'][:15]], dtype=torch.float32).mean()

            self.items.append((torch.stack(imgs), own_speed, tgt_speed))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]



def train_speed_model2():
    train_dir = "validation_crop/train_split"
    val_dir = "validation_crop/val_split"
    annot_dir = "train_annotations"

    train_ds = SpeedDataset(train_dir, annot_dir)
    val_ds = SpeedDataset(val_dir, annot_dir)

    if len(train_ds) == 0:
        raise RuntimeError(f"No training samples found in {train_dir}")

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpeedEstimationModel2().to(device)
    optimizer = Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.MSELoss()

    for epoch in range(10):
        model.train()
        total_loss = 0
        for imgs, speeds, tgts in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, speeds, tgts = imgs.to(device), speeds.to(device), tgts.to(device)
            pred = model(imgs, speeds)
            loss = loss_fn(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        avg_loss = total_loss / len(train_ds)
        print(f"[Epoch {epoch+1}] Train Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "speed_model2.pth")
    print(" モデル保存完了：speed_model2.pth")

    
    model.eval()
    result = {}
    with torch.no_grad():
        for i, (imgs, speeds, tgts) in enumerate(tqdm(val_loader, desc="Evaluating")):
            imgs, speeds = imgs.to(device), speeds.to(device)
            preds = model(imgs, speeds).cpu().numpy().tolist()
            gts = tgts.numpy().tolist()
            for j, (p, g) in enumerate(zip(preds, gts)):
                result[f"sample_{i*16 + j}"] = {"pred": round(p, 2), "gt": round(g, 2)}

    with open("val_result_model2.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print("val_result_model2.json 保存完了")


train_speed_model2()

Epoch 1: 100%|██████████| 36/36 [00:01<00:00, 19.97it/s]


[Epoch 1] Train Loss: 4376.9530


Epoch 2: 100%|██████████| 36/36 [00:00<00:00, 91.39it/s]


[Epoch 2] Train Loss: 4081.1092


Epoch 3: 100%|██████████| 36/36 [00:00<00:00, 91.05it/s]


...

[Epoch 9] Train Loss: 118.1997


Epoch 10: 100%|██████████| 36/36 [00:00<00:00, 91.46it/s]


[Epoch 10] Train Loss: 105.2458
 モデル保存完了：speed_model2.pth


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 205.34it/s]

val_result_model2.json 保存完了


In [5]:
class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc1d = nn.Sequential(
            nn.Conv1d(33, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, 1)
        )

    def forward(self, imgs, speeds):  
        B, T, C, H, W = imgs.shape
        imgs = imgs.view(B * T, C, H, W)
        feats = self.cnn(imgs).view(B, T, -1)  
        speeds = speeds.unsqueeze(-1)  
        concat = torch.cat([feats, speeds], dim=2).permute(0, 2, 1)  
        out = self.fc1d(concat)  
        return out.squeeze(1)


In [10]:


import os
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import Adam
from tqdm import tqdm

class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),  # (1 → 32)
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),  # (32 → 64)
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),  # (64 → 128)
            nn.AdaptiveAvgPool2d((1, 1))  # → (B*T, 128, 1, 1)
        )
        self.fc1d = nn.Sequential(
            nn.Conv1d(129, 128, 3, padding=1), nn.ReLU(),  # 128 from CNN + 1 speed
            nn.Conv1d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, imgs, speeds):
        B, T, C, H, W = imgs.shape
        imgs = imgs.view(B * T, C, H, W)
        feats = self.cnn(imgs).view(B, T, -1)  # → (B, T, 128)
        speeds = speeds.unsqueeze(-1)  # → (B, T, 1)
        concat = torch.cat([feats, speeds], dim=2).permute(0, 2, 1)  # → (B, 129, T)
        out = self.fc1d(concat)
        return out.squeeze(1)




class SpeedDataset(Dataset):
    def __init__(self, crop_root, annot_root):
        self.items = []
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ])
        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue
            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) < 15: continue

            imgs = []
            for i in range(15):
                img_path = os.path.join(crop_dir, files[i])
                img = Image.open(img_path).convert("L")
                imgs.append(self.transform(img))
            if len(imgs) < 15: continue

            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)
                if len(ann['sequence']) < 15: continue
                own_speed = torch.tensor([f['OwnSpeed'] for f in ann['sequence'][:15]], dtype=torch.float32)
                tgt_speed = torch.tensor([f['TgtSpeed_ref'] for f in ann['sequence'][:15]], dtype=torch.float32).mean()

            self.items.append((torch.stack(imgs), own_speed, tgt_speed))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]



def train_speed_model2():
    train_dir = "validation/train_split"
    val_dir = "validation/val_split"
    annot_dir = "train_annotations"

    train_ds = SpeedDataset(train_dir, annot_dir)
    val_ds = SpeedDataset(val_dir, annot_dir)

    if len(train_ds) == 0:
        raise RuntimeError(f"No training samples found in {train_dir}")

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpeedEstimationModel2().to(device)
    optimizer = Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.MSELoss()

    for epoch in range(100):
        model.train()
        total_loss = 0
        for imgs, speeds, tgts in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, speeds, tgts = imgs.to(device), speeds.to(device), tgts.to(device)
            pred = model(imgs, speeds)
            loss = loss_fn(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        avg_loss = total_loss / len(train_ds)
        print(f"[Epoch {epoch+1}] Train Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "speed_model2.pth")
    print(" モデル保存完了：speed_model2.pth")

    
    model.eval()
    result = {}
    with torch.no_grad():
        for i, (imgs, speeds, tgts) in enumerate(tqdm(val_loader, desc="Evaluating")):
            imgs, speeds = imgs.to(device), speeds.to(device)
            preds = model(imgs, speeds).cpu().numpy().tolist()
            gts = tgts.numpy().tolist()
            for j, (p, g) in enumerate(zip(preds, gts)):
                result[f"sample_{i*16 + j}"] = {"pred": round(p, 2), "gt": round(g, 2)}

    with open("val_result_model2.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print("val_result_model2.json 保存完了")


train_speed_model2()

Epoch 1: 100%|██████████| 37/37 [00:01<00:00, 32.73it/s]


[Epoch 1] Train Loss: 4376.2877


Epoch 2: 100%|██████████| 37/37 [00:01<00:00, 34.44it/s]


[Epoch 2] Train Loss: 3190.9480


Epoch 3: 100%|██████████| 37/37 [00:01<00:00, 34.31it/s]


...

[Epoch 99] Train Loss: 205.1596


Epoch 100: 100%|██████████| 37/37 [00:01<00:00, 33.97it/s]


[Epoch 100] Train Loss: 207.1601
 モデル保存完了：speed_model2.pth


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 91.38it/s]

val_result_model2.json 保存完了
